In [1]:
import pandas as pd
import numpy as np

In [2]:
train_labels = pd.read_csv('Data/train.csv')
test_labels = pd.read_csv('Data/dev.csv')

In [3]:
y_train = np.array([label for label in train_labels['PHQ8_Binary']])
y_test = np.array([label for label in test_labels['PHQ8_Binary']])

# Text Model

In [4]:
train_text = []
for label in train_labels['Participant_ID']:
    sample = pd.read_csv(f"Data/train/{label}_P/{label}_TRANSCRIPT.csv", sep = "\t")
    train_text.append(sample)

In [5]:
test_text = []
for label in test_labels['Participant_ID']:
    sample = pd.read_csv(f"Data/test/{label}_P/{label}_TRANSCRIPT.csv", sep = "\t")
    test_text.append(sample)

In [6]:
for sample in train_text:
    sample["total_time"] = sample["stop_time"] - sample["start_time"]
for sample in test_text:
    sample["total_time"] = sample["stop_time"] - sample["start_time"]

In [7]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(train_text[0]["speaker"])
for sample in test_text:
    sample["speaker"] = encoder.transform(sample["speaker"])
for sample in train_text:
    sample["speaker"] = encoder.transform(sample["speaker"])

In [8]:
from sklearn.preprocessing import StandardScaler
Numerics = []
for sample in test_text:
    Numerics.append(sample.loc[:,["start_time", "stop_time", "total_time"]])
Numerics = pd.concat(Numerics)
scale = StandardScaler()
scale.fit(Numerics)
for sample in test_text:
    sample.loc[:,["start_time", "stop_time", "total_time"]] = scale.transform(sample.loc[:,["start_time", "stop_time", "total_time"]])
for sample in train_text:
    sample.loc[:,["start_time", "stop_time", "total_time"]] = scale.transform(sample.loc[:,["start_time", "stop_time", "total_time"]])

In [9]:
from transformers import BertTokenizer, BertModel
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()
model = model.to(device)
def convert(text):
    text = str(text)
    tokens = tokenizer(
    text,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=256
    )
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.no_grad():
        res = model(**tokens)
    return res.last_hidden_state[:, 0, :].detach().cpu().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
for i,sample in enumerate(train_text):
    if sum(sample.isna().sum()) != 0 :
        sample.dropna(inplace=True)
for i,sample in enumerate(test_text):
    if sum(sample.isna().sum()) != 0 :
        sample.dropna(inplace=True)

In [11]:
for sample in train_text:
    sample['value'] = sample['value'].apply(convert)
for sample in test_text:
    sample['value'] = sample['value'].apply(convert)

In [12]:
for i,sample in enumerate(train_text):
    single = np.array([
        np.concatenate([
            np.array([t, s], dtype=np.float32),
            np.asarray(v, dtype=np.float32).ravel()
        ])
        for t, s, v in zip(sample['total_time'], sample['speaker'], sample['value'])
    ], dtype=np.float32)
    train_text[i] = single
for i,sample in enumerate(test_text):
    single = np.array([
        np.concatenate([
            np.array([t, s], dtype=np.float32),
            np.asarray(v, dtype=np.float32).ravel()
        ])
        for t, s, v in zip(sample['total_time'], sample['speaker'], sample['value'])
    ], dtype=np.float32)
    test_text[i] = single

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
train_text = pad_sequences(
    train_text,
    maxlen=473,
    padding='post',      # zeros added at END of each sample
    truncating='post',   # if truncating needed, cut from END
    dtype='float32'
)
test_text = pad_sequences(
    test_text,
    maxlen=473,
    padding='post',      # zeros added at END of each sample
    truncating='post',   # if truncating needed, cut from END
    dtype='float32'
)

In [14]:
from tensorflow.keras.layers import Masking, Bidirectional, LSTM, Dense
from tensorflow.keras.models import Sequential

text = Sequential([
    Masking(mask_value=0.0, input_shape=(473, 770)),
    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])
text.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
text.fit(train_text, y_train, epochs=10, batch_size=4, validation_split=0.2)

D:\Softwears\Lib\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 15s 396ms/step - accuracy: 0.6353 - loss: 0.6632 - val_accuracy: 0.9091 - val_loss: 0.5339
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 359ms/step - accuracy: 0.6706 - loss: 0.6353 - val_accuracy: 0.9091 - val_loss: 0.4721
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 379ms/step - accuracy: 0.6706 - loss: 0.6365 - val_accuracy: 0.9091 - val_loss: 0.4677
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 382ms/step - accuracy: 0.6706 - loss: 0.6309 - val_accuracy: 0.9091 - val_loss: 0.4565
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 371ms/step - accuracy: 0.6706 - loss: 0.6281 - val_accuracy: 0.9091 - val_loss: 0.4506
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 365ms/step - accuracy: 0.6706 - loss: 0.6268 - val_accuracy: 0.9091 - val_loss: 0.4388
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 363ms/step - accuracy: 0.6706 - loss: 0.6204 - val_accuracy: 0.9091 - val_loss: 0.4154
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 363ms/step - accuracy: 0.6706 - loss: 0.6124 - val_accuracy: 0

# Audio Model

In [15]:
train_audio = []
for rank in train_labels['Participant_ID']:
    sample = pd.read_csv(f'Data/train/{rank}_P/{rank}_COVAREP.csv', header=None)
    sample['index'] = sample.index
    time = pd.read_csv(f'Data/train/{rank}_P/{rank}_TRANSCRIPT.csv', sep='\t')
    time = time[time['speaker'] == 'Ellie']
    for start,stop in zip(time['start_time'], time['stop_time']):
        sample.drop(sample[sample['index'].isin(range(int(100*start),int(100*stop)))].index, inplace = True)
    sample = sample[sample[1] == 1]
    train_audio.append(sample.drop(columns = ['index']))

In [16]:
test_audio = []
for rank in test_labels['Participant_ID']:
    sample = pd.read_csv(f'Data/test/{rank}_P/{rank}_COVAREP.csv', header=None)
    sample['index'] = sample.index
    time = pd.read_csv(f'Data/test/{rank}_P/{rank}_TRANSCRIPT.csv', sep='\t')
    time = time[time['speaker'] == 'Ellie']
    for start,stop in zip(time['start_time'], time['stop_time']):
        sample.drop(sample[sample['index'].isin(range(int(100*start),int(100*stop)))].index, inplace = True)
    sample = sample[sample[1] == 1]
    test_audio.append(sample.drop(columns = ['index']))

In [17]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
scale = pd.concat([sample.iloc[:,[0]] for sample in train_audio])
sc.fit(scale)

,copy,True
,with_mean,True
,with_std,True


In [18]:
for sample in train_audio:
    sample.iloc[:,[0]] = sc.transform(sample.iloc[:,[0]])
    sample.drop(columns=sample.columns[1], inplace = True)
for sample in test_audio:
    sample.iloc[:,[0]] = sc.transform(sample.iloc[:,[0]])
    sample.drop(columns=sample.columns[1], inplace = True)

In [19]:
for i,sample in enumerate(train_audio):
    train_audio[i] = sample.groupby(sample.index//25).mean()
for i,sample in enumerate(test_audio):
    test_audio[i] = sample.groupby(sample.index//25).mean()

In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
train_audio = pad_sequences(train_audio, maxlen=3400, dtype='float32', padding='post', value=0.0)
test_audio = pad_sequences(test_audio, maxlen=3400, dtype='float32', padding='post', value=0.0)

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking

audio = Sequential([
    Masking(mask_value=0.0),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
audio.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [22]:
audio.fit(
    train_audio, y_train,
    epochs=10,
    batch_size=4,
    validation_split=0.2
)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.6471 - loss: 0.6581 - val_accuracy: 0.9091 - val_loss: 0.4864
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.6824 - loss: 0.6138 - val_accuracy: 0.9091 - val_loss: 0.4716
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.6941 - loss: 0.6116 - val_accuracy: 0.7727 - val_loss: 0.5155
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.7059 - loss: 0.5823 - val_accuracy: 0.7273 - val_loss: 0.4870
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.7176 - loss: 0.5356 - val_accuracy: 0.7273 - val_loss: 0.5059
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.7412 - loss: 0.5041 - val_accuracy: 0.5455 - val_loss: 0.6152
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.7647 - loss: 0.5049 - val_accuracy: 0.6818 - val_loss: 0.4712
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.7059 - loss: 0.5091 - val_accuracy: 0.7273 - val_loss:

# Video Model

In [23]:
train_video = []
for add in train_labels['Participant_ID']:
    au = pd.read_csv(f'Data/train/{add}_P/{add}_CLNF_AUs.txt')
    gaze = pd.read_csv(f'Data/train/{add}_P/{add}_CLNF_gaze.txt')
    pose = pd.read_csv(f'Data/train/{add}_P/{add}_CLNF_pose.txt')
    sample = au.merge(gaze).merge(pose)
    sample = sample[sample[' confidence'] > 0.75]
    sample = sample[sample[' success'] == 1]
    sample.drop(columns=[ 'frame', ' timestamp', ' confidence', ' success'], inplace=True)
    train_video.append(sample)

In [24]:
test_video = []
for add in test_labels['Participant_ID']:
    au = pd.read_csv(f'Data/test/{add}_P/{add}_CLNF_AUs.txt')
    gaze = pd.read_csv(f'Data/test/{add}_P/{add}_CLNF_gaze.txt')
    pose = pd.read_csv(f'Data/test/{add}_P/{add}_CLNF_pose.txt')
    sample = au.merge(gaze).merge(pose)
    sample = sample[sample[' confidence'] > 0.75]
    sample = sample[sample[' success'] == 1]
    sample.drop(columns=[ 'frame', ' timestamp', ' confidence', ' success'], inplace=True)
    test_video.append(sample)

In [25]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
sc.fit(pd.concat(train_video, axis=0, ignore_index=True))
for i in range(len(train_video)):
    train_video[i] = sc.transform(train_video[i])
for i in range(len(test_video)):
    test_video[i] = sc.transform(test_video[i])

In [26]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_video = pad_sequences(train_video, maxlen=52256, padding='post', value=0.0, dtype='float32')
test_video = pad_sequences(test_video, maxlen=52256, padding='post', value=0.0, dtype='float32')

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Masking, Dropout, LSTM

video = Sequential([
    Masking(mask_value=0.0),
    LSTM(64, input_shape=(train_video.shape[1], train_video.shape[2])),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

video.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

D:\Softwears\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [33]:
video.fit(
    train_video, y_train,
    epochs=10,
    batch_size=4,
    validation_split=0.2
)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 346s 16s/step - accuracy: 0.7059 - loss: 0.6493 - val_accuracy: 0.6364 - val_loss: 0.6544
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 380s 17s/step - accuracy: 0.7529 - loss: 0.5774 - val_accuracy: 0.7273 - val_loss: 0.6247
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 376s 17s/step - accuracy: 0.8118 - loss: 0.5108 - val_accuracy: 0.7273 - val_loss: 0.6194
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 421s 19s/step - accuracy: 0.8353 - loss: 0.4698 - val_accuracy: 0.6818 - val_loss: 0.6003
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 385s 17s/step - accuracy: 0.9059 - loss: 0.3988 - val_accuracy: 0.6364 - val_loss: 0.5981
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 385s 18s/step - accuracy: 0.8941 - loss: 0.3569 - val_accuracy: 0.6818 - val_loss: 0.5934
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 388s 18s/step - accuracy: 0.9059 - loss: 0.3237 - val_accuracy: 0.5909 - val_loss: 0.6183
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 392s 18s/step - accuracy: 0.9176 - loss: 0.2707 - val_accuracy: 0.

# Predict

In [34]:
text_pred = text.predict(test_text)
audio_pred = audio.predict(test_audio)
video_pred = video.predict(test_video)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 301ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step


In [67]:
avg_pred = (text_pred + audio_pred + video_pred) / 3
y_avg = [1 if p > 0.62 else 0 for p in avg_pred]

In [68]:
from sklearn.metrics import confusion_matrix, accuracy_score
print(confusion_matrix(y_test, y_avg))
print(accuracy_score(y_test, y_avg))

[[23  0]
 [12  0]]
0.6571428571428571


In [55]:
max_pred = np.maximum(np.maximum( text_pred, audio_pred), video_pred)
y_max = [1 if p > 0.44 else 0 for p in max_pred]

In [56]:
print(confusion_matrix(y_test, y_max))
print(accuracy_score(y_test, y_max))

[[12 11]
 [ 4  8]]
0.5714285714285714
